In [ ]:
import requests
import json
import os
from dotenv import load_dotenv

# ==========================================
# 1. TUS CREDENCIALES
# ==========================================
load_dotenv()
CLIENT_ID = os.getenv(CLIENT_ID)
CLIENT_SECRET = os.getenv(CLIENT_SECRET)

# ==========================================
# 2. OBTENER EL TOKEN DE AUTENTICACIÓN
# ==========================================
def obtener_token():
    token_endpoint = 'http://icdaccessmanagement.who.int/connect/token'
    payload = {
        'grant_type': 'client_credentials',
        'scope': 'icdapi_access'
    }
    
    print("Obteniendo token de la OMS...")
    # La API de la OMS usa Basic Auth para entregar el token
    response = requests.post(token_endpoint, data=payload, auth=(CLIENT_ID, CLIENT_SECRET))
    
    if response.status_code == 200:
        return response.json()['access_token']
    else:
        raise Exception(f"Error al obtener token: {response.status_code} - {response.text}")

# ==========================================
# 3. BUSCAR Y EXTRAER DATOS EN ESPAÑOL
# ==========================================
def buscar_en_cie11(termino_busqueda, token):
    # Endpoint de búsqueda en la versión MMS (la versión clínica/estadística estándar)
    # mms = Mortality and Morbidity Statistics
    # icf = Clasificación Internacional del Funcionamiento, de la Discapacidad y de la Salud
    search_url = f'http://localhost/icd/release/11/2026-01/mms/search?q={termino_busqueda}'
    
    # ¡ESTOS HEADERS SON LA CLAVE!
    headers = {
        'Authorization': f'Bearer {token}',
        'Accept': 'application/json',
        'Accept-Language': 'es', # <--- Esto fuerza que los títulos y sinónimos vengan en español
        'API-Version': 'v2'
    }
        
    print(f"Buscando: '{termino_busqueda}'...")
    response = requests.get(search_url, headers=headers)
    
    if response.status_code == 200:
        resultados = response.json()
        
        # Procesamos los resultados para extraer lo que le sirve a tu RAG
        entidades_procesadas = []
        for destino in resultados.get('destinationEntities', []):
            codigo = destino.get('theCode', 'Sin Código')
            titulo = destino.get('title', '').replace('<em>', '').replace('</em>', '') # Limpiar HTML
            
            # Extraer sinónimos (matching terms)
            sinonimos = []
            for matching_term in destino.get('matchingPVs', []):
                term = matching_term.get('label', '').replace('<em>', '').replace('</em>', '')
                if term and term not in sinonimos:
                    sinonimos.append(term)
            
            entidades_procesadas.append({
                'codigo': codigo,
                'titulo_oficial': titulo,
                'sinonimos': sinonimos
            })
            
        return entidades_procesadas
    else:
        print(f"Error en la búsqueda: {response.status_code}")
        return []

# ==========================================
# EJECUCIÓN DEL FLUJO
# ==========================================
if __name__ == "__main__":
    lista = []
    try:
        # 1. Autenticarse
        mi_token = obtener_token()
        
        # 2. Hacer una prueba buscando "Colico renal"
        search_url = f'http://localhost/icd/release/11/2026-01/mms/474718032'
        #'child': ['http://localhost/icd/release/11/2026-01/mms/1435254666', 'http://localhost/icd/release/11/2026-01/mms/1630407678', 'http://localhost/icd/release/11/2026-01/mms/1766440644', 'http://localhost/icd/release/11/2026-01/mms/1954798891', 'http://localhost/icd/release/11/2026-01/mms/21500692', 'http://localhost/icd/release/11/2026-01/mms/334423054', 'http://localhost/icd/release/11/2026-01/mms/274880002', 'http://localhost/icd/release/11/2026-01/mms/1296093776', 'http://localhost/icd/release/11/2026-01/mms/868865918', 'http://localhost/icd/release/11/2026-01/mms/1218729044', 'http://localhost/icd/release/11/2026-01/mms/426429380', 'http://localhost/icd/release/11/2026-01/mms/197934298', 'http://localhost/icd/release/11/2026-01/mms/1256772020', 'http://localhost/icd/release/11/2026-01/mms/1639304259', 'http://localhost/icd/release/11/2026-01/mms/1473673350', 'http://localhost/icd/release/11/2026-01/mms/30659757', 'http://localhost/icd/release/11/2026-01/mms/577470983', 'http://localhost/icd/release/11/2026-01/mms/714000734', 'http://localhost/icd/release/11/2026-01/mms/1306203631', 'http://localhost/icd/release/11/2026-01/mms/223744320', 'http://localhost/icd/release/11/2026-01/mms/1843895818', 'http://localhost/icd/release/11/2026-01/mms/435227771', 'http://localhost/icd/release/11/2026-01/mms/850137482', 'http://localhost/icd/release/11/2026-01/mms/1249056269', 'http://localhost/icd/release/11/2026-01/mms/1596590595', 'http://localhost/icd/release/11/2026-01/mms/718687701', 'http://localhost/icd/release/11/2026-01/mms/231358748', 'http://localhost/icd/release/11/2026-01/mms/979408586']
        
        #search_url = f'http://localhost/icd/release/11/2026-01/icf'
        #'child': ['http://localhost/icd/release/11/2026-01/icf/619527855', 'http://localhost/icd/release/11/2026-01/icf/423829389']
    
        # ¡ESTOS HEADERS SON LA CLAVE!
        headers = {
            'Authorization': f'Bearer {mi_token}',
            'Accept': 'application/json',
            'Accept-Language': 'es', # <--- Esto fuerza que los títulos y sinónimos vengan en español
            'API-Version': 'v2'
        }
        response = requests.get(search_url, headers=headers)
        if response.status_code == 200:
            print("Búsqueda exitosa. Procesando resultados...")
            resultados = response.json()
            print(resultados)
            lista.append(resultados)
            print(lista)
            nombre_archivo = 'muestra2.json'
            with open(nombre_archivo, 'w', encoding='utf-8') as archivo:
            # ensure_ascii=False permite que las tildes y ñ se guarden correctamente
            # indent=4 le da el formato bonito con saltos de línea
                json.dump(lista, archivo, ensure_ascii=False, indent=4)
            """
            resultados = response.json()
            resultados = resultados.get('child', []) 
            for child in resultados:
                child = child.replace('http://id.who.int', 'http://localhost')
                r = requests.get(child, headers=headers)
                if r.status_code == 200:
                    lista.append(r.json())
                    #print(r.json())
                else:
                    print(f"Error al obtener detalles de {child}: {r.status_code}")
            print("Resultados obtenidos correctamente.")
            print(resultados)
            nombre_archivo = 'muestra.json'
            with open(nombre_archivo, 'w', encoding='utf-8') as archivo:
            # ensure_ascii=False permite que las tildes y ñ se guarden correctamente
            # indent=4 le da el formato bonito con saltos de línea
                json.dump(lista, archivo, ensure_ascii=False, indent=4)
            """
        else:
            print(f"Error en la búsqueda: {response.status_code}")
        """
        resultados = buscar_en_cie11("colico renal", mi_token)
        
        # 3. Imprimir los resultados listos para tu base vectorial
        print("\n--- RESULTADOS PARA EL RAG ---")
        for idx, item in enumerate(resultados[:3]): # Mostramos solo los 3 primeros
            print(f"\nOpción {idx + 1}:")
            print(f"Código CIE-11: {item['codigo']}")
            print(f"Título:        {item['titulo_oficial']}")
            print(f"Sinónimos:     {', '.join(item['sinonimos'])}")
        """
            
    except Exception as e:
        print(e)


Obteniendo token de la OMS...
Búsqueda exitosa. Procesando resultados...
{'@context': 'http://id.who.int/icd/contexts/contextForLinearizationEntity.json', '@id': 'http://id.who.int/icd/release/11/2026-01/mms/474718032', 'parent': ['http://id.who.int/icd/release/11/2026-01/mms/1294458332'], 'browserUrl': 'https://icd.who.int/browse/2026-01/mms/es#474718032', 'code': 'DD51', 'source': 'http://id.who.int/icd/entity/474718032', 'classKind': 'category', 'postcoordinationScale': [{'@id': 'http://id.who.int/icd/release/11/2026-01/mms/474718032/postcoordinationScale/laterality', 'axisName': 'http://id.who.int/icd/schema/laterality', 'requiredPostcoordination': 'false', 'allowMultipleValues': 'NotAllowed', 'scaleEntity': ['http://id.who.int/icd/release/11/2026-01/mms/627678743', 'http://id.who.int/icd/release/11/2026-01/mms/271422288', 'http://id.who.int/icd/release/11/2026-01/mms/876572005', 'http://id.who.int/icd/release/11/2026-01/mms/1038788978']}, {'@id': 'http://id.who.int/icd/release/11/

¡Es una duda buenísima y súper lógica! Suena a una contradicción gigante: *"Si mi usuario va a escribir como se le dé la gana (texto libre), ¿de qué me sirve buscar la palabra exacta?"*

El malentendido aquí suele venir de lo que imaginamos cuando decimos "búsqueda exacta". Déjame explicarte por qué la búsqueda híbrida (Hybrid Search) es precisamente la solución perfecta para el texto libre.

### El mito de la "Búsqueda Exacta"

Cuando escuchamos "búsqueda exacta", solemos pensar en la lógica de las bases de datos tradicionales (como un `WHERE diagnóstico = 'Hernia inguinal directa'`). Si el texto libre es *"paciente refiere que tiene una hernia inguinal directa desde ayer"*, ese tipo de búsqueda exacta fallaría porque las frases no son idénticas.

Sin embargo, en el mundo de la Búsqueda Híbrida, la parte "exacta" (llamada *Sparse Retrieval* o algoritmo **BM25**) **no busca que toda la frase sea igual, sino que busca coincidencias de palabras clave individuales dentro de ese texto libre.**

### Ejemplo práctico: El choque entre lo semántico y lo exacto

Imagina que el médico escribe este diagnóstico en texto libre:
> *"Paciente de 45 años ingresa con cuadro compatible con **diabetes tipo 1** descompensada."*

Veamos cómo actúan los dos motores por separado y por qué se necesitan mutuamente:

#### 1. Solo Embeddings (Búsqueda Semántica / Dense)
* **Cómo funciona:** La IA lee todo el texto, entiende el "concepto" (enfermedad metabólica, azúcar, descompensación) y lo convierte en coordenadas matemáticas.
* **El problema:** Para una IA, los conceptos de "diabetes tipo 1" y "diabetes tipo 2" están **matemáticamente casi en el mismo lugar**. Ambas son enfermedades metabólicas, comparten casi las mismas palabras y el mismo contexto. 
* **El resultado:** El buscador semántico podría devolverte el código de la "Diabetes tipo 2" en primer lugar solo porque estadísticamente es más común en sus datos de entrenamiento, ignorando por completo el número "1".

#### 2. Solo Palabras Clave (Algoritmo BM25 / Sparse)
* **Cómo funciona:** Destruye la frase en pedazos ("paciente", "ingresa", "diabetes", "tipo", "1") y busca cuántas de esas palabras raras y específicas coinciden con tus `indexTerms`.
* **El problema:** Si el médico escribe *"diabetis"* (con error ortográfico) o *"azúcar alta"*, el buscador de palabras clave no encontrará nada porque literalmente esa palabra no está en tu base de datos.
* **El resultado:** Es muy rígido ante errores humanos o sinónimos.

### La magia del híbrido en el texto libre

Al combinar ambos motores, creas un sistema a prueba de balas para el texto libre:

1. El usuario envía el texto libre sucio y desordenado.
2. **El motor semántico (Vectores)** dice: *"Por el contexto general, esto trata sobre hernias en la zona abdominal"* (y te trae un grupo de códigos de hernias, perdonando faltas de ortografía o jergas).
3. **El motor de palabras clave (BM25)** revisa ese texto libre buscando anclas exactas y dice: *"¡Ojo! En medio de todo ese texto libre, el médico escribió la palabra **'indirecta'**. Así que dale un puntaje extra a la Hernia Inguinal Indirecta"*.

**En resumen:** No estás obligando al usuario a escribir la patología exacta. Le estás permitiendo escribir un párrafo de texto libre de 5 líneas, y usando el motor de palabras clave (BM25) para "escanear" si dentro de toda esa libertad el usuario mencionó **números, siglas o apellidos específicos** (como Tipo 1, Tipo 2, VIH, COVID) que a la inteligencia artificial (los vectores) se le suelen escapar por ser demasiado generalista.

In [5]:
import requests
import json
import time

CLIENT_ID = '899f4f48-e50e-44cd-9243-d93fcd806f77_84830157-bb14-4ad0-bae1-bc0ef2ddf215'
CLIENT_SECRET = 'AsV/FO/TntGaRYQwLs8BEHt0Rz10mTtXW8awcENXDck='
# ==========================================
# 0. OBTENER EL TOKEN
# ==========================================
def obtener_token():
    token_endpoint = 'http://icdaccessmanagement.who.int/connect/token'
    payload = {
        'grant_type': 'client_credentials',
        'scope': 'icdapi_access'
    }
    
    print("Obteniendo token de la OMS...")
    # La API de la OMS usa Basic Auth para entregar el token
    response = requests.post(token_endpoint, data=payload, auth=(CLIENT_ID, CLIENT_SECRET))
    
    if response.status_code == 200:
        return response.json()['access_token']
    else:
        raise Exception(f"Error al obtener token: {response.status_code} - {response.text}")


# ==========================================
# 1. FUNCIÓN RECURSIVA
# ==========================================
def extraer_jerarquia(url, headers, lista_maestra):
    """
    Función que extrae los datos de una URL y luego se llama a sí misma 
    para extraer los datos de todos sus hijos.
    """
    print(f"Consultando nodo: {url.split('/')[-1]}...") # Imprime solo el ID final para no saturar la pantalla
    url = url.replace('http://id.who.int', 'http://localhost')
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        nodo = response.json()
        
        # --- A. EXTRAER LOS DATOS DEL NODO ACTUAL ---
        codigo = nodo.get('code', '') # Algunos nodos "carpeta" no tienen código oficial
        
        # El título en la API suele venir como {"@language": "es", "@value": "Nombre"}
        titulo_obj = nodo.get('title', {})
        titulo = titulo_obj.get('@value', '') if isinstance(titulo_obj, dict) else str(titulo_obj)
        
        # Extraer sinónimos (Index Terms)
        sinonimos = []
        for term in nodo.get('indexTerm', []):
            label = term.get('label', {})
            valor_sinonimo = label.get('@value', '') if isinstance(label, dict) else str(label)
            if valor_sinonimo:
                sinonimos.append(valor_sinonimo)
                
        # Guardar en nuestra lista de memoria (solo si tiene un título válido)
        if titulo:
            lista_maestra.append({
                "codigo": codigo,
                "titulo": titulo,
                "sinonimos": sinonimos,
                "url": url
            })

        # --- B. LA MAGIA RECURSIVA (BUSCAR HIJOS) ---
        hijos = nodo.get('child', []) 
        
        for child_url in hijos:
            # ⚠️ MUY IMPORTANTE: Esperar 0.2 segundos entre peticiones.
            # Si haces miles de peticiones por segundo sin pausa, 
            # la OMS detectará un ataque y bloqueará tu IP.
            time.sleep(0.2) 
            
            # La función se llama a sí misma con la nueva URL del hijo
            extraer_jerarquia(child_url, headers, lista_maestra)
            
    else:
        print(f"Error al obtener {url}: {response.status_code}")

# ==========================================
# 2. EJECUCIÓN Y GUARDADO EN JSON
# ==========================================
if __name__ == "__main__":
    # Asegúrate de poner tu token real aquí
    TOKEN = obtener_token()
    
    mis_headers = {
        'Authorization': f'Bearer {TOKEN}',
        'Accept': 'application/json',
        'Accept-Language': 'es', 
        'API-Version': 'v2'
    }
    
    # Lista vacía donde la función recursiva irá metiendo todos los diccionarios
    datos_completos_cie11 = []
    
    # ⚠️ CONSEJO DE ORO: No empieces desde la raíz total la primera vez.
    # La CIE-11 es inmensa y tardará horas. Prueba primero con una sub-categoría 
    # para asegurar que tu código funciona. 
    # Aquí probaremos con un nodo específico (ej. "Enfermedades del apéndice")
    URL_INICIAL = 'http://localhost/icd/release/11/2026-01/mms/1435254666'
    
    print("Iniciando extracción recursiva...")
    
    try:
        # Llamamos a la función por primera vez
        extraer_jerarquia(URL_INICIAL, mis_headers, datos_completos_cie11)
        
        print(f"\n¡Extracción terminada! Se obtuvieron {len(datos_completos_cie11)} enfermedades.")
        
        # --- C. GUARDAR TODO EN UN ARCHIVO JSON ---
        nombre_archivo = 'datos_cie11.json'
        with open(nombre_archivo, 'w', encoding='utf-8') as archivo:
            # ensure_ascii=False permite que las tildes y ñ se guarden correctamente
            # indent=4 le da el formato bonito con saltos de línea
            json.dump(datos_completos_cie11, archivo, ensure_ascii=False, indent=4)
            
        print(f"Los datos se han guardado exitosamente en el archivo: {nombre_archivo}")

    except Exception as e:
        print(f"Se produjo un error durante la ejecución: {e}")

Obteniendo token de la OMS...
Iniciando extracción recursiva...
Consultando nodo: 1435254666...
Consultando nodo: 588616678...
Consultando nodo: 135352227...
Consultando nodo: 257068234...
Consultando nodo: 416025325...
Consultando nodo: 2080365623...
Consultando nodo: 344162786...
Consultando nodo: 2099226249...
Consultando nodo: 408185629...
Consultando nodo: 1828273122...
Consultando nodo: 162723448...
Consultando nodo: other...
Consultando nodo: unspecified...
Consultando nodo: 250688797...
Consultando nodo: 1000894786...
Consultando nodo: 794462570...
Consultando nodo: 1528414070...
Consultando nodo: 364534567...
Consultando nodo: other...
Consultando nodo: unspecified...
Consultando nodo: 1780040028...
Consultando nodo: 515117475...
Consultando nodo: 1520312138...
Consultando nodo: other...
Consultando nodo: unspecified...
Consultando nodo: other...
Consultando nodo: unspecified...
Consultando nodo: 1834648119...
Consultando nodo: 1642556936...
Consultando nodo: 78422942...
Consu